In [ ]:
import kagglehub
path=kagglehub.dataset_download("prachi13/customer-analytics")
print("Path to dataset files:",path)

Using Colab cache for faster access to the 'customer-analytics' dataset.
Path to dataset files: /kaggle/input/customer-analytics


In [ ]:
import pandas as pd
df=pd.read_csv("/kaggle/input/customer-analytics/Train.csv")

1) Linear Regression (predict Final_Price)

In [ ]:
# Linear Regression - predicting Final_Price
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# Ensure Final_Price exists
df['Final_Price'] = df['Cost_of_the_Product'] - df['Discount_offered']

# Features and target
X = df.drop(columns=['ID', 'Final_Price', 'Reached.on.Time_Y.N'])  # drop ID and target & delivery flag
y = df['Final_Price']

# Categorical and numeric columns
cat_cols = ['Warehouse_block', 'Mode_of_Shipment', 'Product_importance', 'Gender']
num_cols = [c for c in X.columns if c not in cat_cols]

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Preprocessing
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_cols)
])

# Pipeline with Linear Regression
pipe = Pipeline([
    ('pre', preprocessor),
    ('lr', LinearRegression())
])

# Train
pipe.fit(X_train, y_train)

# Predict & evaluate
y_pred = pipe.predict(X_test)
print("Linear Regression metrics:")
print("  RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("  MAE:", mean_absolute_error(y_test, y_pred))
print("  R2:", r2_score(y_test, y_pred))

Linear Regression metrics:
  RMSE: 5.948686066322515e-14
  MAE: 4.6973334313158626e-14
  R2: 1.0


2) Logistic Regression (predict Reached.on.Time_Y.N)

In [ ]:
# Logistic Regression - predicting Reached.on.Time_Y.N
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score

# Target
y = df['Reached.on.Time_Y.N']
X = df.drop(columns=['ID', 'Reached.on.Time_Y.N'])  # drop ID and target

# Columns
cat_cols = ['Warehouse_block', 'Mode_of_Shipment', 'Product_importance', 'Gender']
num_cols = [c for c in X.columns if c not in cat_cols]

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Preprocessing
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_cols)
])

# Pipeline with Logistic Regression
pipe_clf = Pipeline([
    ('pre', preprocessor),
    ('logreg', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
])

# Train
pipe_clf.fit(X_train, y_train)

# Predict & evaluate
y_pred = pipe_clf.predict(X_test)
y_proba = pipe_clf.predict_proba(X_test)[:,1]

print("Logistic Regression metrics:")
print("  Accuracy:", accuracy_score(y_test, y_pred))
print("  Precision:", precision_score(y_test, y_pred))
print("  Recall:", recall_score(y_test, y_pred))
print("  F1-score:", f1_score(y_test, y_pred))
print("  ROC-AUC:", roc_auc_score(y_test, y_proba))
print("  Confusion matrix:\n", confusion_matrix(y_test, y_pred))

Logistic Regression metrics:
  Accuracy: 0.6454545454545455
  Precision: 0.8277982779827798
  Recall: 0.5125666412795126
  F1-score: 0.6331138287864534
  ROC-AUC: 0.7169901883085716
  Confusion matrix:
 [[747 140]
 [640 673]]


SVM Classifier (predict Reached.on.Time_Y.N)

In [13]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix

# Target and features
y = df['Reached.on.Time_Y.N']
X = df.drop(columns=['ID', 'Reached.on.Time_Y.N'])

# Separate categorical and numerical columns
cat_cols = ['Warehouse_block', 'Mode_of_Shipment', 'Product_importance', 'Gender']
num_cols = [c for c in X.columns if c not in cat_cols]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Preprocessor
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_cols)
])

# Build pipeline
svc_pipe = Pipeline([
    ('preprocess', preprocessor),
    ('classifier', SVC(C=1.0, kernel='rbf', probability=True, random_state=42))
])

# Cross-validation on training set
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(svc_pipe, X_train, y_train, cv=cv, scoring='f1')

print("Cross-validation F1 scores:", cv_scores)
print("Mean CV F1-score:", np.mean(cv_scores))

# Fit final model
svc_pipe.fit(X_train, y_train)

# Predictions
y_pred = svc_pipe.predict(X_test)

print("\nClassification Report (Test Set):")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))


Cross-validation F1 scores: [0.67074527 0.66471963 0.65088757 0.66191781 0.66392479]
Mean CV F1-score: 0.6624390150473008

Classification Report (Test Set):
              precision    recall  f1-score   support

           0       0.56      0.87      0.68       887
           1       0.86      0.53      0.66      1313

    accuracy                           0.67      2200
   macro avg       0.71      0.70      0.67      2200
weighted avg       0.74      0.67      0.67      2200

Confusion Matrix:
[[769 118]
 [612 701]]
